# Week 15, Day 2: Base cases and making progress

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w015d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def halvings(n):
    if n == 1:
        return 0
    return 1 + halvings(n // 2)

print(halvings(8), halvings(10), halvings(1))

In [ ]:
def total(items):
    if len(items) == 1:
        return items[0]
    return items[0] + total(items[1:])

print(total([3, 4, 5]))
print(total([]))

In [ ]:
def total(items):
    if not items:
        return 0
    return items[0] + total(items[1:])

print(total([3, 4, 5]), total([]))

> **Key idea.** Choose the smallest possible input as the base case (0, the empty list, the empty string), and its natural answer. It usually covers the edge cases for free.

In [ ]:
def collatz(n):
    if n == 1:
        return [1]
    nxt = n // 2 if n % 2 == 0 else 3 * n + 1
    return [n] + collatz(nxt)

print(collatz(6))

In [ ]:
def factorial(n):
    if not isinstance(n, int) or n < 0:
        raise ValueError(f"factorial needs a whole number of 0 or more, got {n!r}")
    if n == 0:
        return 1
    return n * factorial(n - 1)

print(factorial(5))
for bad in (-3, 2.5):
    try:
        factorial(bad)
    except ValueError as e:
        print(e)

In [ ]:
def factorial(n):
    if not isinstance(n, int) or n < 0:
        raise ValueError(f"factorial needs a whole number of 0 or more, got {n!r}")
    return _factorial(n)

def _factorial(n):
    return 1 if n == 0 else n * _factorial(n - 1)

print(factorial(6))

In [ ]:
def fib(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fib(n - 1) + fib(n - 2)

print([fib(n) for n in range(10)])

In [ ]:
def is_even(n):
    if n == 0:
        return True
    return is_odd(n - 1)

def is_odd(n):
    if n == 0:
        return False
    return is_even(n - 1)

print(is_even(10), is_odd(7), is_even(3))

In [ ]:
import sys
print(sys.getrecursionlimit())

def depth(n):
    return 0 if n == 0 else 1 + depth(n - 1)

print(depth(500))
try:
    depth(5000)
except RecursionError as e:
    print("RecursionError:", e)

> **Tip.** `RecursionError` almost always means one of two things: a missing or unreachable base case (the common bug), or recursion one level per item on a very long input (the design problem). Check which before anything else.

### Exercise: Count a character

Write `count(text, ch)` that returns how many times `ch` appears in `text`, recursively. Base case: the empty string. Recursive case: the first character, and the rest.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert count(\"banana\", \"a\") == 3 and count(\"\", \"a\") == 0 and count(\"xyz\", \"a\") == 0\n    assert \"for \" not in source and \"while \" not in source and \".count(\" not in source")

### Exercise: The largest

Write `largest(items)` that returns the largest item of a list recursively, without `max`. Raise a `ValueError` for an empty list. (Careful: what is the base case here, if the empty list has no answer?)

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert largest([3, 9, 4]) == 9 and largest([-5, -2, -9]) == -2 and largest([7]) == 7\n    try:\n        largest([])\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"an empty list should raise a ValueError\")\n    assert \"max(\" not in source and \"for \" not in source and \"while \" not in source")

### Exercise: A guarded power

Write `power(x, n)` for a whole number `n` of 0 or more, raising a `ValueError` for a negative or non-whole `n`, so it never recurses for ever. Use an outer function that checks and a recursive helper.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"powers and guards\"\n    assert power(2, 10) == 1024 and power(7, 0) == 1 and power(1.5, 2) == 2.25\n    for bad in (-1, 2.5, \"3\"):\n        try:\n            power(2, bad)\n        except ValueError:\n            continue\n        raise AssertionError(f\"power(2, {bad!r}) should raise a ValueError\")\n    assert \"for \" not in source and \"while \" not in source and \"**\" not in source")

### Exercise: Faster powers

`power(x, n)` above makes n calls. A cleverer recursive case halves n instead: if n is even, x to the n is (x to the n/2) squared; if odd, it's x times (x to the n - 1). Write `fast_power(x, n)` for n of 0 or more this way. What shrinks now, and how fast? (`fast_power(2, 1000)` makes about 15 calls instead of 1,000.)

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"powers\"\n    assert [fast_power(2, n) for n in range(12)] == [2 ** n for n in range(12)]\n    assert fast_power(3, 200) == 3 ** 200 and fast_power(2, 5000) == 2 ** 5000\n    assert \"for \" not in source and \"while \" not in source and \"**\" not in source and \"pow(\" not in source")

### Exercise: Three base cases

The **tribonacci** numbers start 0, 0, 1, and each one after that is the sum of the three before it: 0, 0, 1, 1, 2, 4, 7, 13, 24, ... Write `trib(n)` recursively. How many base cases does it need?

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert [trib(n) for n in range(10)] == [0, 0, 1, 1, 2, 4, 7, 13, 24, 44]\n    assert \"for \" not in source and \"while \" not in source")

### Exercise: Mutual recursion: Hofstadter's sequences

In his book *Gödel, Escher, Bach* (1979), Douglas Hofstadter defined two sequences in terms of each other:

```
F(0) = 1,  M(0) = 0
F(n) = n - M(F(n - 1))     for n > 0
M(n) = n - F(M(n - 1))     for n > 0
```

Write `F(n)` and `M(n)`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert [F(n) for n in range(12)] == [1, 1, 2, 2, 3, 3, 4, 5, 5, 6, 6, 7]\n    assert [M(n) for n in range(12)] == [0, 0, 1, 2, 2, 3, 4, 4, 5, 6, 6, 7]")

### Bug hunt: One base case short

`fib(n)` raises `RecursionError` for every n except 0. Fix it.

In [ ]:
def fib(n):
    if n == 0:
        return 0
    return fib(n - 1) + fib(n - 2)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"fib\"\n    assert [fib(n) for n in range(10)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]")

### Bug hunt: The wrong base case

`join_words(words)` should join a list of words with single spaces, like `" ".join(words)`: `["a", "b", "c"]` gives `"a b c"`. It gives `"a b c "`, with a stray space, and for `[]` it should give `""`. Fix the base case.

In [ ]:
def join_words(words):
    if not words:
        return ""
    return words[0] + " " + join_words(words[1:])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert join_words([\"a\", \"b\", \"c\"]) == \"a b c\"\n    assert join_words([\"solo\"]) == \"solo\"\n    assert join_words([]) == \"\"\n    assert \".join(\" not in source")

### Puzzle: Is it sorted?

Arrange the lines so `is_sorted(items)` checks recursively whether a list is in increasing order. One line is a trap.

(Put these lines in order on the website.)

### Exercise: What shrinks?

For each recursive call, choose what shrinks: `"n"`, `"length"`, `"digits"`, or `"nothing"` (it might never finish).

In [ ]:
a = ___   # def f(n): return 0 if n == 0 else f(n - 1)
b = ___   # def g(s): return "" if not s else g(s[1:]) + s[0]
c = ___   # def h(n): return 1 if n == 1 else h(n + 1)
d = ___   # def k(n): return 0 if n < 10 else 1 + k(n // 10)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (a, b, c, d) == (\"n\", \"length\", \"nothing\", \"digits\")")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: counting the ways

How many ways are there to climb a staircase of `n` steps, if each move goes up 1, 2 or 3 steps? For 3 steps there are 4 ways: 1+1+1, 1+2, 2+1 and 3. Write `ways(n)` recursively, and then `routes(n)` that returns the list of every way, each as a list of moves, in the order: those starting with 1 first, then 2, then 3 (and so on inside each). Think carefully about the base cases: how many ways are there to climb 0 steps? And what about negative steps?

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"ways\"\n    assert [ways(n) for n in range(8)] == [1, 1, 2, 4, 7, 13, 24, 44]\ndef test_routes():\n    \"routes\"\n    assert routes(3) == [[1, 1, 1], [1, 2], [2, 1], [3]]\n    assert routes(0) == [[]]\n    assert len(routes(6)) == ways(6)\n    assert all(sum(r) == 6 for r in routes(6))")

## Warm-up

Now think about `halvings(0)`. `0 // 2` is 0, so it calls `halvings(0)` again, for ever. The base case `n == 1` is right for every positive number, but nothing stops a 0 getting in. That's today's subject.

## Choosing the base case

The base case is the **smallest problem** the function can be asked to solve. Choose it by asking: what's the simplest input, and what's the answer?

- For a list: usually the **empty list**. Its sum is 0, its length is 0, its reverse is itself.
- For a string: usually the **empty string**.
- For a count n: usually **0**, sometimes 1.

It's tempting to choose a bigger base case, like "a list of one item", because it feels more natural. But then the empty list either crashes or recurses for ever:

With the empty list as the base case, there's nothing to forget:

`items[0]` is the **first** item, and `items[1:]` is **the rest**. "The first item, and a smaller list of the rest" is the most common way to make a list smaller.

## Making progress

For a recursive function to finish, each recursive call must be on a **smaller** problem, and "smaller" must eventually reach the base case. Name the quantity that shrinks:

| Function | What shrinks | Reaches the base case? |
|---|---|---|
| `factorial(n - 1)` | n, by 1 | yes, from any n ≥ 0 |
| `total(items[1:])` | the list's length, by 1 | yes, an empty list |
| `digits(n // 10)` | the number of digits, by 1 | yes, below 10 |
| `gcd(b, a % b)` | the second number (`a % b < b`) | yes, 0 |
| `halvings(n // 2)` | n, roughly halved | from 1 or more, yes; from 0, never |

When you write a recursive function, check this table's question for it: **what shrinks, and does it always reach the base case?** If you can't say, the function might not finish.

`3 * n + 1` makes n **bigger**. So nothing obviously shrinks, and in fact nobody has been able to prove this function finishes for every starting number. It does for every number anyone has tried (into the quintillions), but that's an unsolved problem in mathematics, not something you'd want in your own code.

## Guarding the inputs

If some inputs can never reach the base case, decide what should happen to them (Week 14's edge cases again). Usually, raise an error **before** recursing:

Without the guard, `factorial(-3)` would recurse until `RecursionError`, and `factorial(2.5)` would skip straight past 0 to -0.5, -1.5, ... for ever. The check happens on every call, which is a little wasteful; a common alternative is a small "outer" function that checks once, and calls a recursive helper.

The leading underscore in `_factorial` is a convention meaning "a helper, not meant to be called directly".

## Several base cases, and mutual recursion

Some functions need more than one base case, because the recursive case reaches back more than one step. The **Fibonacci numbers** are 0, 1, 1, 2, 3, 5, 8, ..., where each is the sum of the previous two:

`fib(n)` calls `fib(n - 2)`, so with only the base case `n == 0`, `fib(1)` would call `fib(-1)` and never stop. Each recursive step down needs a base case to land on. (This version is also very slow for large n, because it works out the same values again and again. You'll fix that next week.)

Functions can also be recursive **through each other**. This is called **mutual recursion**:

It's a silly way to test for even numbers, but the shape matters: later you'll see parsers and games where two or three functions call each other naturally, following the structure of the problem.

## The recursion limit

Each waiting call takes memory, for its frame. To stop a runaway recursion from using up all the memory and crashing, Python allows about **1,000** frames at once, and raises `RecursionError` beyond that:

So in Python, recursion that goes one level deeper per item, like `total(items[1:])`, is fine for hundreds of items but fails for tens of thousands. `sys.setrecursionlimit` can raise the limit, but the real answer for long sequences is usually a loop. You'll compare the two on Day 5.

## Practice

Solve these recursively, with no loops.

## Quiz

## Daily challenge

## Before you go

For each recursive function you wrote today, say out loud: the base case, what shrinks, and why it always reaches the base case. Tomorrow: the call stack, frame by frame, so you can see exactly what Python is doing, and why recursion can sometimes be slow.